# Bài 12 — Fine-tune **BERT** trên **GLUE/SST-2** + Baseline **TransformerEncoder**

**Tác vụ**: Phân loại cảm xúc nhị phân (positive/negative) trên SST-2.  
**Môi trường**: `transformers`, `datasets`, `torch`, `scikit-learn`, `matplotlib`.  
**Ràng buộc**: `epoch ≤ 5`, `batch_size = 32`, `lr ∈ {2e-5, 3e-5}`, `weight_decay = 0.01`, `max_len = 128`, `fp16 nếu có`.

> Lưu ý: Nhãn **test** của GLUE không công khai. Đo lường chính xác sẽ dùng **validation**; phần **test** chỉ sinh file dự đoán để nộp (nếu cần).


## 0. Cài đặt & cấu hình
- Nếu đã có các gói bên dưới, có thể bỏ qua cell pip.
- Khuyên dùng GPU (CUDA) để đạt batch size 32 theo ràng buộc.


In [ ]:

# (Tuỳ chọn) Cài đặt thư viện — bỏ qua nếu đã có
# !pip install -U transformers datasets accelerate evaluate scikit-learn matplotlib

import os, random, math, time, json, numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (AutoTokenizer, DataCollatorWithPadding,
                          BertForSequenceClassification, TrainingArguments, Trainer,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Tham số chung theo đề bài
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 5  # ≤ 5
LR_GRID = [2e-5, 3e-5]
WEIGHT_DECAY = 0.01
MODEL_NAME = "bert-base-uncased"

OUTPUT_DIR = "./Data/BERT_SST2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FP16_FLAG = bool(torch.cuda.is_available())  # fp16 nếu có
print("Use fp16:", FP16_FLAG)


## 1. Nạp dữ liệu GLUE/SST-2 và tokenize

In [ ]:

raw = load_dataset("glue", "sst2")
print(raw)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    return tokenizer(batch["sentence"],
                     truncation=True,
                     max_length=MAX_LEN,
                     padding="max_length")

encoded = raw.map(preprocess, batched=True, remove_columns=["sentence"])

# Cột đầu vào cần cho Trainer
encoded = encoded.rename_column("label", "labels")
encoded.set_format(type="torch",
                   columns=["input_ids", "token_type_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=None)


## 2. Hàm đánh giá (Accuracy & ROC-AUC) và tiện ích vẽ ma trận nhầm lẫn

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = torch.tensor(logits)
    probs = torch.softmax(preds, dim=-1).cpu().numpy()
    y_pred = probs.argmax(axis=-1)
    y_true = labels

    acc = accuracy_score(y_true, y_pred)

    # ROC-AUC nhị phân: dùng xác suất lớp dương (label=1)
    try:
        auc = roc_auc_score(y_true, probs[:, 1])
    except Exception as e:
        print("ROC-AUC error:", e)
        auc = float("nan")

    return {"accuracy": acc, "roc_auc": auc}

def plot_confusion(y_true, y_pred, labels=("neg","pos")):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(4, 4))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title("Confusion Matrix (dev)")
    plt.tight_layout()
    return fig


## 3. Huấn luyện BERT (`bert-base-uncased`) với **Trainer** + grid LR {2e-5, 3e-5}

In [ ]:

def train_eval_bert(lr):
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/bert-lr{lr}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        fp16=FP16_FLAG,
        logging_steps=50,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=encoded["train"],
        eval_dataset=encoded["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_res = trainer.evaluate()
    print("Eval:", eval_res)

    # Confusion matrix trên dev
    preds = trainer.predict(encoded["validation"])
    y_true = preds.label_ids
    y_prob = torch.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
    y_pred = y_prob.argmax(axis=-1)

    fig = plot_confusion(y_true, y_pred, labels=("neg","pos"))
    fig.savefig(f"{args.output_dir}/confusion_matrix_dev.png")
    plt.close(fig)

    # Lưu ROC điểm số
    with open(f"{args.output_dir}/metrics_dev.json","w") as f:
        json.dump({"eval": eval_res}, f, indent=2)

    # Trả về để so sánh
    return {"lr": lr, "metrics": eval_res, "outdir": args.output_dir}

results = []
for lr in LR_GRID:
    print(f"=== Training with lr = {lr} ===")
    res = train_eval_bert(lr)
    results.append(res)

# Chọn theo accuracy tốt nhất
best = max(results, key=lambda x: x["metrics"].get("eval_accuracy", float("-inf")))
print("Best setting:", best)
with open(os.path.join(OUTPUT_DIR, "grid_results.json"), "w") as f:
    json.dump(results, f, indent=2)


## 4. Sinh dự đoán **test** (không có nhãn) để nộp GLUE (tuỳ chọn)

In [ ]:
best_dir = "./Data/BERT_SST2/bert-lr3e-05/checkpoint-10525"

# Ép tải model ở dtype float32 (an toàn cho mọi GPU)
model = BertForSequenceClassification.from_pretrained(
    best_dir,
    torch_dtype=torch.float32,     # ép kiểu float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("✅ Model loaded successfully on", device)


In [ ]:
def export_test_predictions(best_dir):
    """
    Sinh file dự đoán test (GLUE SST-2) từ checkpoint tốt nhất.
    - Không cần labels, chỉ xuất index & prediction.
    """
    # đảm bảo không có labels trong test
    test_ds = encoded["test"]
    if "labels" in test_ds.column_names:
        test_ds = test_ds.remove_columns("labels")

    trainer = Trainer(model=model, tokenizer=tokenizer, data_collator=data_collator)
    preds = trainer.predict(test_ds)

    # softmax -> argmax
    y_prob = torch.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
    y_pred = y_prob.argmax(axis=-1)

    # chỉ số index (bắt buộc cho GLUE)
    idxs = raw["test"]["idx"]
    assert len(idxs) == len(y_pred)

    out_path = os.path.join(best_dir, "sst2-test-pred.tsv")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("index\tprediction\n")
        for i, p in zip(idxs, y_pred):
            f.write(f"{i}\t{int(p)}\n")

    print(f"✅ Wrote test predictions to: {out_path}")

# chạy
export_test_predictions(best_dir)


## 5. Baseline: **TransformerEncoder** tự xây (không tiền huấn luyện)

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=2, dim_ff=512, num_labels=2, max_len=MAX_LEN):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, batch_first=True)
        self.enc = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls = nn.Linear(d_model, num_labels)

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.emb(input_ids)  # (B, T, D)
        x = self.pos(x)
        if attention_mask is not None:
            # PyTorch Transformer uses True for attend; convert 0/1 -> True/False mask
            key_padding_mask = (attention_mask == 0)  # (B, T)
        else:
            key_padding_mask = None
        h = self.enc(x, src_key_padding_mask=key_padding_mask)
        # Dùng embedding vị trí [CLS]-style: lấy vector vị trí 0
        pooled = h[:, 0, :]
        logits = self.cls(pooled)
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

def run_tiny_transformer(train_ds, val_ds, vocab_size, lr=2e-4, epochs=5):
    model = TinyTransformerClassifier(vocab_size=vocab_size).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    best = {"acc": -1, "state": None}

    def make_loader(ds, shuffle):
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

    train_loader = make_loader(train_ds, shuffle=True)
    val_loader = make_loader(val_ds, shuffle=False)

    for ep in range(1, epochs+1):
        model.train()
        for batch in train_loader:
            for k in batch:
                batch[k] = batch[k].to(device)
            out = model(batch["input_ids"], batch["attention_mask"], batch["labels"])
            out["loss"].backward()
            optim.step()
            optim.zero_grad()

        # eval
        model.eval()
        y_true, y_prob = [], []
        with torch.no_grad():
            for batch in val_loader:
                for k in batch:
                    batch[k] = batch[k].to(device)
                out = model(batch["input_ids"], batch["attention_mask"])
                logits = out["logits"]
                probs = torch.softmax(logits, dim=-1)
                y_prob.append(probs[:,1].detach().cpu())
                y_true.append(batch["labels"].detach().cpu())
        y_true = torch.cat(y_true).numpy()
        y_prob = torch.cat(y_prob).numpy()
        y_pred = (y_prob >= 0.5).astype(int)

        acc = accuracy_score(y_true, y_pred)
        try:
            auc = roc_auc_score(y_true, y_prob)
        except Exception:
            auc = float("nan")
        print(f"[TinyTransformer] Epoch {ep}: acc={acc:.4f}, roc_auc={auc:.4f}")
        if acc > best["acc"]:
            best = {"acc": acc, "auc": auc, "state": model.state_dict()}

    # Vẽ confusion matrix ở epoch tốt nhất (sau epoch cuối cùng)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=("neg","pos"))
    fig, ax = plt.subplots(figsize=(4,4))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title("Confusion Matrix (dev) — TinyTransformer")
    plt.tight_layout()
    figpath = os.path.join(OUTPUT_DIR, "tiny_transformer_confusion_dev.png")
    fig.savefig(figpath)
    plt.close(fig)
    print("Saved:", figpath)

    return best

# Chạy baseline (có thể tốn thời gian trên CPU)
# best_tiny = run_tiny_transformer(encoded["train"], encoded["validation"], vocab_size=tokenizer.vocab_size, lr=2e-4, epochs=EPOCHS)
# print("Best TinyTransformer:", best_tiny)


## 6. Tuỳ chọn: **Đóng băng** các lớp đầu & **Layer-wise LR Decay (LLRD)** cho BERT

In [ ]:

def freeze_bert_layers(model, num_frozen_layers=6):
    # Đóng băng embeddings + N encoder layers đầu
    for p in model.bert.embeddings.parameters():
        p.requires_grad = False
    for layer in model.bert.encoder.layer[:num_frozen_layers]:
        for p in layer.parameters():
            p.requires_grad = False
    return model

def get_llrd_param_groups(model, base_lr, lr_decay=0.95):
    # Tạo param groups với LR giảm dần từ trên xuống dưới encoder
    # Nhóm head (classifier) dùng base_lr
    params = []
    # Classifier
    params.append({"params": [p for p in model.classifier.parameters() if p.requires_grad], "lr": base_lr})
    # Encoder layers (last -> first)
    L = len(model.bert.encoder.layer)
    cur_lr = base_lr
    for i in range(L-1, -1, -1):
        layer = model.bert.encoder.layer[i]
        layer_params = [p for p in layer.parameters() if p.requires_grad]
        if layer_params:
            params.append({"params": layer_params, "lr": cur_lr})
        cur_lr *= lr_decay
    # Embeddings (nếu không bị đóng băng)
    emb_params = [p for p in model.bert.embeddings.parameters() if p.requires_grad]
    if emb_params:
        params.append({"params": emb_params, "lr": cur_lr})
    return params

# Ví dụ sử dụng (tắt mặc định vì nâng cao):
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = freeze_bert_layers(model, num_frozen_layers=6)
param_groups = get_llrd_param_groups(model, base_lr=2e-5, lr_decay=0.95)
optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
# Tính tổng bước train để tạo scheduler
steps_per_epoch = math.ceil(len(encoded["train"]) / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)
trainer = Trainer(model=model,
                  args=TrainingArguments(output_dir=f"{OUTPUT_DIR}/bert-llrd",
                                         per_device_train_batch_size=BATCH_SIZE,
                                         per_device_eval_batch_size=BATCH_SIZE,
                                         num_train_epochs=EPOCHS,
                                         evaluation_strategy="epoch",
                                         save_strategy="epoch",
                                         load_best_model_at_end=True,
                                         metric_for_best_model="accuracy",
                                         learning_rate=2e-5,
                                         fp16=FP16_FLAG,
                                         weight_decay=WEIGHT_DECAY,
                                         report_to="none",
                                         seed=SEED),
                  train_dataset=encoded["train"],
                  eval_dataset=encoded["validation"],
                  tokenizer=tokenizer,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  optimizers=(optimizer, scheduler))
trainer.train()
print(trainer.evaluate())


## 7. Tuỳ chọn: Data Augmentation (EDA) cho câu ngắn (SSR/RI/RS/RD)

In [ ]:

# EDA đơn giản (không cần thư viện ngoài). Có thể cải thiện bằng textattack/nlpaug nếu muốn.
import re, random

def eda_synonym_replacement(tokens, n=1):
    # Thay thế ngẫu nhiên n từ (bỏ stopword thô sơ)
    words = [w for w in tokens if w.isalpha() and len(w) > 3]
    if not words: 
        return tokens
    for _ in range(n):
        idxs = [i for i,t in enumerate(tokens) if t in words]
        if not idxs: break
        i = random.choice(idxs)
        tokens[i] = tokens[i]  # placeholder: không có từ đồng nghĩa -> giữ nguyên
    return tokens

def eda_random_insertion(tokens, n=1):
    for _ in range(n):
        if tokens:
            i = random.randrange(0, len(tokens)+1)
            tokens.insert(i, tokens[max(0, min(len(tokens)-1, i))])
    return tokens

def eda_random_swap(tokens, n=1):
    for _ in range(n):
        if len(tokens) > 1:
            i, j = random.sample(range(len(tokens)), 2)
            tokens[i], tokens[j] = tokens[j], tokens[i]
    return tokens

def eda_random_deletion(tokens, p=0.1):
    if len(tokens) == 1: return tokens
    return [t for t in tokens if random.random() > p] or tokens[:1]

def eda_augment_sentence(text):
    tokens = text.split()
    if len(tokens) < 3: 
        return text
    ops = [lambda t: eda_synonym_replacement(t, n=1),
           lambda t: eda_random_insertion(t, n=1),
           lambda t: eda_random_swap(t, n=1),
           lambda t: eda_random_deletion(t, p=0.1)]
    random.shuffle(ops)
    aug = tokens[:]
    for op in ops[:2]:  # áp 2 phép bất kỳ
        aug = op(aug)
    return " ".join(aug)

# Ví dụ áp dụng trên một phần tập train (để tránh phình to dataset quá mức)
# USE_EDA = False
# if USE_EDA:
#     import pandas as pd
#     train_texts = raw["train"]["sentence"]
#     train_labels = raw["train"]["label"]
#     n_aug = int(0.2 * len(train_texts))  # augment 20%
#     idxs = np.random.choice(len(train_texts), size=n_aug, replace=False)
#     aug_texts = [eda_augment_sentence(train_texts[i]) for i in idxs]
#     aug_labels = [train_labels[i] for i in idxs]
#     # Hợp lại vào Dataset (đơn giản hoá bằng map từ list)
#     from datasets import Dataset
#     ds_aug = Dataset.from_dict({"sentence": aug_texts, "label": aug_labels})
#     raw_train_aug = Dataset.from_dict({"sentence": train_texts, "label": train_labels}).concatenate(ds_aug)
#     # Tokenize lại
#     train_encoded = raw_train_aug.map(preprocess, batched=True, remove_columns=["sentence"]).rename_column("label","labels")
#     train_encoded.set_format(type="torch", columns=["input_ids","token_type_ids","attention_mask","labels"])
#     # Dùng 'train_encoded' thay cho 'encoded["train"]' khi huấn luyện


## 8. Tổng kết & so sánh
- Sau khi chạy xong, xem:
  - `outputs_sst2/bert-lr*/confusion_matrix_dev.png`
  - `outputs_sst2/grid_results.json`
  - (Tuỳ chọn) `outputs_sst2/bert-llrd/*` nếu chạy nâng cao.
- Bảng kết quả có thể tổng hợp thủ công từ log/JSON.
